# Nero LoRA Training 🧠
Trenuje LoRA adapter na wspomnieniach Nero.
**Wymagane: A100 GPU** (Runtime → Change runtime type → A100)


In [ ]:
# 1. Instalacja
!pip install -q unsloth
!pip install -q trl datasets
print('Instalacja gotowa!')


In [ ]:
# 2. Wgraj dataset.jsonl
from google.colab import files
print('Wybierz plik dataset.jsonl z dysku lokalnego:')
uploaded = files.upload()
print('Wgrano:', list(uploaded.keys()))


In [ ]:
# 3. Zaladuj model Gemma 3 27B (4-bit dla oszczednosci VRAM)
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/gemma-3-27b-it-bnb-4bit',
    max_seq_length=1024,
    load_in_4bit=True,
    dtype=None,
)
print('Model zaladowany!')


In [ ]:
# 4. Dodaj LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('LoRA gotowa!')


In [ ]:
# 5. Przygotuj dataset
import json
from datasets import Dataset

def format_prompt(row):
    return (f"<start_of_turn>system\n{row['system']}<end_of_turn>\n"
            f"<start_of_turn>user\n{row['instruction']}<end_of_turn>\n"
            f"<start_of_turn>model\n{row['output']}<end_of_turn>")

records = []
with open('dataset.jsonl', encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line))

dataset = Dataset.from_list([{'text': format_prompt(r)} for r in records])
print(f'Dataset: {len(dataset)} probek treningowych')


In [ ]:
# 6. Trening (~20-30 min na A100)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=1024,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        output_dir='nero_lora_output',
        warmup_ratio=0.1,
        lr_scheduler_type='cosine',
        report_to='none',
    ),
)
trainer.train()
print('Trening zakonczony!')


In [ ]:
# 7. Eksport do GGUF
model.save_pretrained_gguf('nero_lora_gguf', tokenizer, quantization_method='q5_k_m')
print('Adapter GGUF gotowy!')

# Pobierz
import os
from google.colab import files
for fname in os.listdir('nero_lora_gguf'):
    if fname.endswith('.gguf'):
        print(f'Pobieranie: {fname}')
        files.download(f'nero_lora_gguf/{fname}')
